## Training models
Requires a zip file containing preprocessed images into *AN2DL_Challenge2-TheBigBatchTheory/data/processed*.

### Preamble
Drive connection, setup for fast loading, Github repo connection

In [1]:
from google.colab import drive
import os, sys, random, subprocess, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import time
!pip install -q comet_ml torchsummary
from comet_ml import start
from comet_ml.integration.pytorch import log_model
from torch import nn
from torchsummary import summary
from torch.utils.tensorboard import SummaryWriter
import cv2
from PIL import Image
from tqdm.notebook import tqdm

# 1. Mount Drive
drive.mount('/content/drive', force_remount=True)

logs_dir = "tensorboard"
!pkill -f tensorboard
%load_ext tensorboard
!mkdir -p models

# 2. Setup Seed e Device
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 3. Setup Giuthub Repository
REPO = "an2dl-challenges-25-26"
TARGET_FOLDER = "challenge2"  # <--- YOUR SUBFOLDER NAME

# Clone repo
if os.path.exists(REPO):
    print(f"Folder '{REPO}' already exists. Deleting it for a fresh clone...")
    shutil.rmtree(REPO)  # Recursively deletes the folder and its contents

# Now clone it fresh
!git clone https://github.com/asarraa/{REPO}.git

# 3. Navigate INTO the subfolder
# We construct the full path: /content/an2dl-challenges-25-26/challenge2
project_path = os.path.abspath(os.path.join(os.getcwd(), REPO, TARGET_FOLDER))

# Add to Python Path (so imports work)
if project_path not in sys.path:
    sys.path.append(project_path)

# Change Directory (so internal file references work)
os.chdir(project_path)

print(f"Setup Github complete.")
print(f"Current Working Directory: {os.getcwd()}")

# --------------------------------------------------------------------------
# 4. SETUP DATI VELOCI (DRIVE -> LOCALE SSD)
# --------------------------------------------------------------------------
DATA_VARIANT = 'preprocess_v1_weighted'  # Il nome della variante/zip
DRIVE_ROOT = Path('/content/drive/MyDrive/AN2DL_Challenge2-TheBigBatchTheory/')
DRIVE_DATA_ROOT = DRIVE_ROOT / 'data/preprocessed'

# TODO: semplificare il path, il dataset processato sta in DRIVE_DATA_ROOT/{DATA_VARIANT}.zip
zip_path = DRIVE_DATA_ROOT / f"{DATA_VARIANT}.zip"

LOCAL_DATA_ROOT = Path('/content/local_data')

if zip_path:
    print(f"Trovato zip in: {zip_path}")
    print("Copia ed estrazione in corso su SSD locale (richiede ~30-60 sec)...")

    # Copia in temp per velocità
    temp_zip = Path("/content/temp_dataset.zip")
    shutil.copy(zip_path, temp_zip)

    # Estrai
    shutil.unpack_archive(temp_zip, LOCAL_DATA_ROOT)

    # Pulisci
    os.remove(temp_zip)
    print("Estrazione completata!")
else:
    raise FileNotFoundError(f"ERRORE: Non trovo {DATA_VARIANT}.zip su Drive!\nControlla in: {DRIVE_DATA_ROOT}")

# --------------------------------------------------------------------------
# 5. DEFINIZIONE PERCORSI (BASE_PATH ora punta al LOCALE)
# --------------------------------------------------------------------------

# Gestiamo il caso in cui lo zip contenga una cartella col nome della variante o meno
if (LOCAL_DATA_ROOT / DATA_VARIANT).exists():
    BASE_PATH = LOCAL_DATA_ROOT / DATA_VARIANT
else:
    BASE_PATH = LOCAL_DATA_ROOT

TRAIN_IMG_DIR = BASE_PATH / 'train/images'
#TRAIN_MSK_DIR = BASE_PATH / 'train' / 'masks'
TEST_IMG_DIR = BASE_PATH / 'test/images'
#TEST_MSK_DIR = BASE_PATH / 'test' / 'masks'
TRAIN_CSV_PATH = BASE_PATH / 'train/train_patches.csv'

# Verifiche di sicurezza
assert TRAIN_IMG_DIR.exists(), f'Errore: Cartella immagini non trovata in {TRAIN_IMG_DIR}'
assert TRAIN_CSV_PATH.exists(), f'Errore: CSV non trovato in {TRAIN_CSV_PATH}'

print(f'Using device: {device}')
print(f'DATASET PRONTO IN LOCALE: {BASE_PATH}')

Mounted at /content/drive
Folder 'an2dl-challenges-25-26' already exists. Deleting it for a fresh clone...
Cloning into 'an2dl-challenges-25-26'...
remote: Enumerating objects: 627, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 627 (delta 119), reused 105 (delta 56), pack-reused 455 (from 1)
Receiving objects: 100% (627/627), 16.80 MiB | 17.61 MiB/s, done.
Resolving deltas: 100% (427/427), done.
Setup Github complete.
Current Working Directory: /content/an2dl-challenges-25-26/challenge2
Trovato zip in: /content/drive/MyDrive/AN2DL_Challenge2-TheBigBatchTheory/data/preprocessed/preprocess_v1_weighted.zip
Copia ed estrazione in corso su SSD locale (richiede ~30-60 sec)...
Estrazione completata!
Using device: cuda
DATASET PRONTO IN LOCALE: /content/local_data/preprocess_v1_weighted


### Dataloaders

In [2]:
import lazy_loaders

BATCH_SIZE = 128
ADD_MASK_CHANNEL = False

train_loader, val_loader, input_shape, class_weights = lazy_loaders.get_loaders(
    batch_size=BATCH_SIZE,
    add_mask_channel=ADD_MASK_CHANNEL,
    base_path=BASE_PATH,
)

test_loader, _ = lazy_loaders.get_test_loaders(
    batch_size=BATCH_SIZE,
    add_mask_channel=ADD_MASK_CHANNEL,
    base_path=BASE_PATH,
)


Train samples: 12125, Val samples: 3032
Input shape: (3, 224, 224)
Test samples: 12477, Input shape: (3, 224, 224)


### Training

In [3]:
from launch_training import start_training
from models import EfficientNetModel

MODEL_NAME = 'CNN'
TRAINING_PARAMS = {
    'epochs': 1,
    'patience': 10,
}

trained_model, history, exp_id = start_training(
    model_name=MODEL_NAME,
    training_params=TRAINING_PARAMS,
    train_loader=train_loader,
    val_loader=val_loader,
    data_input_shape=input_shape,
    local_data_path=BASE_PATH,
    device=device,
    class_weights=class_weights,
)


[DEBUG] Device type: <class 'torch.device'>, Final device: cuda, CUDA available: True
Using GPU: Tesla T4
--- Starting CNN on cuda ---


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.


✓ Updated input_shape to: (3, 224, 224)
Starting CNN model training...
Training Configuration:
 epochs: 1
learning_rate: 0.001
patience: 10
l1_lambda: 0
l2_lambda: 0
verbose: 10
criterion_name: CrossEntropyLoss
optimizer_name: adamw
Model Configuration:
 input_shape: (3, 224, 224)
num_classes: 4
num_blocks: 2
convs_per_block: 1
use_stride: False
stride_value: 2
padding_size: 1
pool_size: 2
initial_channels: 32
channel_multiplier: 2
dropout_rate_classifier_head: 0.2


COMET INFO: Experiment is live on comet.com https://www.comet.com/asarraa/test/94013c5dadf44ea6a8bf8772a2dd471e



[DEBUG] About to instantiate model...
[DEBUG] Initializing CNN model with the following parameters:
input_shape: (3, 224, 224)
num_classes: 4
num_blocks: 2
convs_per_block: 1
use_stride: False
stride_value: 2
padding_size: 1
pool_size: 2
initial_channels: 32
channel_multiplier: 2
dropout_rate_classifier_head: 0.2
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 224, 224]             896
              ReLU-2         [-1, 32, 224, 224]               0
         MaxPool2d-3         [-1, 32, 112, 112]               0
   VanillaCNNBlock-4         [-1, 32, 112, 112]               0
            Conv2d-5         [-1, 64, 112, 112]          18,496
              ReLU-6         [-1, 64, 112, 112]               0
         MaxPool2d-7           [-1, 64, 56, 56]               0
   VanillaCNNBlock-8           [-1, 64, 56, 56]               0
           Flatten-9               [-1, 2007

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:659: UserWarning: pin_memory_device is deprecated, the current accelerator will be used as the device,ignore pin_memory_device='cuda'.
  warnings.warn(


[DEBUG] Processing first batch, shape: torch.Size([128, 3, 224, 224])
[DEBUG] Forward pass done, logits shape: torch.Size([128, 4])
[DEBUG] First batch complete
[DEBUG] Processed 10/95 batches
[DEBUG] Processed 20/95 batches
[DEBUG] Processed 30/95 batches
[DEBUG] Processed 40/95 batches
[DEBUG] Processed 50/95 batches
[DEBUG] Processed 60/95 batches
[DEBUG] Processed 70/95 batches
[DEBUG] Processed 80/95 batches
[DEBUG] Processed 90/95 batches
[DEBUG] All batches processed, computing metrics...
Epoch 1 train done


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:659: UserWarning: pin_memory_device is deprecated, the current accelerator will be used as the device,ignore pin_memory_device='cuda'.
  warnings.warn(


Epoch   1/1 | Train: Loss=1.4318, F1 Score=0.3049 | Val: Loss=1.3284, F1 Score=0.2031
Best model restored from epoch 0 with val_f1 0.2031
Model saved to: /content/local_data/preprocess_v1_weighted/experiments/models/CNN_20251211_183200.pt
Registry updated: ID CNN_20251211_183200


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : CNN_20251211_183200
COMET INFO:     url                   : https://www.comet.com/asarraa/test/94013c5dadf44ea6a8bf8772a2dd471e
COMET INFO:   Metrics:
COMET INFO:     F1/Training     : 0.3048762048258621
COMET INFO:     F1/Validation   : 0.20312594584314286
COMET INFO:     Loss/Training   : 1.4318270828797646
COMET INFO:     Loss/Validation : 1.3283947457738792
COMET INFO:     train_f1        : 0.3048762048258621
COMET INFO:     train_loss      : 1.4318270828797646
COMET INFO:     val_f1          : 0.20312594584314286
COMET INFO:     val_loss        : 1.3283947457738792
COMET INFO:   Others:
COMET INFO:     Name                : CNN_20251211_183200
C

In [4]:
# TODO: check the function
def grid_search():

  learning_rates = [1e-3]
  batch_sizes = [256] # Match the BATCH_SIZE you defined in Data Processing

  for lr, batch in product(learning_rates, batch_sizes):

      print(f"\n>>> Running Experiment: LR={lr}, Batch={batch}")

      # We pass train_loader, val_loader, and input_shape explicitly!
      trained_model, history, exp_id = start_training(
          device = device,
          train_loader=train_loader,   # <--- Defined in your Notebook cells
          val_loader=val_loader,       # <--- Defined in your Notebook cells
          data_input_shape=input_shape,     # <--- Defined in your Notebook cells
          model_name="CNN",
          training_params={
              "batch_size": 256,
          }
      )


In [4]:
# Train 5 splits for ensembling. Toggle RUN_ENSEMBLE_TRAINING to control execution.
from pathlib import Path
from inference import make_ensemble_inference

RUN_ENSEMBLE_TRAINING = True
SEED_LIST = [11, 22, 33, 44, 55]
ENSEMBLE_TRAINING_PARAMS = {
    'epochs': 1,
    'patience': 10,
}

ensemble_runs = []

if RUN_ENSEMBLE_TRAINING:
    for split_idx, split_seed in enumerate(SEED_LIST, start=1):
        print(f"\n>>> Split {split_idx}/{len(SEED_LIST)} with seed {split_seed}")
        lazy_loaders.SEED = split_seed
        train_loader, val_loader, input_shape, class_weights = lazy_loaders.get_loaders(
            batch_size=BATCH_SIZE,
            add_mask_channel=ADD_MASK_CHANNEL,
            base_path=BASE_PATH,
        )

        _, _, exp_id = start_training(
            model_name=MODEL_NAME,
            training_params=ENSEMBLE_TRAINING_PARAMS,
            train_loader=train_loader,
            val_loader=val_loader,
            data_input_shape=input_shape,
            local_data_path=BASE_PATH,
            device=device,
            class_weights=class_weights,
        )

        model_path = BASE_PATH / 'experiments/models' / f"{exp_id}.pt"
        ensemble_runs.append({
            'seed': split_seed,
            'exp_id': exp_id,
            'model_path': model_path,
            'model_name': MODEL_NAME,
        })
        print(f"Saved ensemble member {split_idx}: {model_path}")
else:
    print("Skipping ensemble training loop. Set RUN_ENSEMBLE_TRAINING = True to enable it.")


### Saving on Drive

In [5]:
# @title
'''# This is the step that replaces "Pushing to GitHub"
import datetime# Create a timestamped folder name so you don't overwrite old runs
DRIVE_SAVE_PATH = DRIVE_ROOT / 'Runs'
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
save_dir = DRIVE_SAVE_PATH / f"Run_{timestamp}" #os.path.join(DRIVE_SAVE_PATH, f"Run_{timestamp}")
save_dir.mkdir(parents=True, exist_ok=True)#print(f"Saving results in: {save_dir}")# 1. Copy the Registry/Final Models
if Path("experiments").exists():
    shutil.copytree("experiments", save_dir / "experiments")# 2. Copy the Checkpoints (Safety Net)
if Path("models").exists():
    shutil.copytree("models", save_dir / "models")print(f"Saved to Drive in {save_dir}.")'''

'# This is the step that replaces "Pushing to GitHub"\nimport datetime# Create a timestamped folder name so you don\'t overwrite old runs\nDRIVE_SAVE_PATH = DRIVE_ROOT / \'Runs\'\ntimestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")\nsave_dir = DRIVE_SAVE_PATH / f"Run_{timestamp}" #os.path.join(DRIVE_SAVE_PATH, f"Run_{timestamp}")\nsave_dir.mkdir(parents=True, exist_ok=True)#print(f"Saving results in: {save_dir}")# 1. Copy the Registry/Final Models\nif Path("experiments").exists():\n    shutil.copytree("experiments", save_dir / "experiments")# 2. Copy the Checkpoints (Safety Net)\nif Path("models").exists():\n    shutil.copytree("models", save_dir / "models")print(f"Saved to Drive in {save_dir}.")'

In [6]:
# @title
'''import os
import shutil
import subprocess
from pathlib import Path
from google.colab import userdata

# 1. Setup Authentication (Required for Push)
# For Google Colab, you need to provide your GitHub Token directly.
# Replace 'YOUR_GITHUB_TOKEN_HERE' with your actual token.
# You can generate a Personal Access Token (PAT) from GitHub settings -> Developer settings -> Personal access tokens -> Tokens (classic).
# Ensure your PAT has 'repo' scope.
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN') # <<< IMPORTANT: REPLACE WITH YOUR TOKEN

if GITHUB_TOKEN == "YOUR_GITHUB_TOKEN_HERE":
    print("⚠️ Warning: GITHUB_TOKEN is not set. Please replace 'YOUR_GITHUB_TOKEN_HERE' with your actual GitHub Personal Access Token. Git push will be skipped.")
else:
    try:
        # Configure Git User (Required for Commit)
        # Replace with your actual email and name
        !git config --global user.email "colab-user@example.com"
        !git config --global user.name "Colab User"

        # Update Remote URL with Token to allow Pushing
        # We strip the protocol to insert the token safely
        repo_url = f"https://{GITHUB_TOKEN}@github.com/asarraa/{REPO}"
        !git remote set-url origin {repo_url}
        print("✅ Git authentication setup complete.")

    except Exception as e:
        print(f"❌ Error setting up Git authentication: {e}")

# 2. Define Paths
# Source: Where the training script saved it (in the writable temp data)
source_registry = BASE_PATH / "experiments/registry.json"

# Destination: The 'experiments' folder in your current Git repo folder
dest_folder = Path("experiments")
dest_registry = dest_folder / "registry.json"

# 3. Move/Copy the file
if source_registry.exists():
    # Create local experiments folder if it doesn't exist
    dest_folder.mkdir(parents=True, exist_ok=True)

    # Copy the file (Using copy instead of move is safer for re-runs)
    shutil.copy(str(source_registry), str(dest_registry))
    print(f"✅ Copied registry.json to {dest_registry}")

    # 4. Git Operations
    print("Running Git commands...")

    # Only attempt Git operations if GITHUB_TOKEN is set
    if GITHUB_TOKEN != "YOUR_GITHUB_TOKEN_HERE":
        try:
            # Add the file
            !git add experiments/registry.json

            # Commit
            !git commit -m "registry modification"

            # Push
            # Note: Change 'main' to 'master' if your repo uses master branch
            !git push origin main

            print("🚀 Successfully pushed to GitHub!")

        except subprocess.CalledProcessError as e:
            print(f"❌ Git Error: {e}")
    else:
        print("🚫 Skipping Git push: GITHUB_TOKEN not set.")
else:
    print(f"❌ Error: Source file not found at {source_registry}")'''

'import os\nimport shutil\nimport subprocess\nfrom pathlib import Path\nfrom google.colab import userdata\n\n# 1. Setup Authentication (Required for Push)\n# For Google Colab, you need to provide your GitHub Token directly.\n# Replace \'YOUR_GITHUB_TOKEN_HERE\' with your actual token.\n# You can generate a Personal Access Token (PAT) from GitHub settings -> Developer settings -> Personal access tokens -> Tokens (classic).\n# Ensure your PAT has \'repo\' scope.\nGITHUB_TOKEN = userdata.get(\'GITHUB_TOKEN\') # <<< IMPORTANT: REPLACE WITH YOUR TOKEN\n\nif GITHUB_TOKEN == "YOUR_GITHUB_TOKEN_HERE":\n    print("⚠️ Warning: GITHUB_TOKEN is not set. Please replace \'YOUR_GITHUB_TOKEN_HERE\' with your actual GitHub Personal Access Token. Git push will be skipped.")\nelse:\n    try:\n        # Configure Git User (Required for Commit)\n        # Replace with your actual email and name\n        !git config --global user.email "colab-user@example.com"\n        !git config --global user.name "Colab 


### Inference on test set

In [16]:
from inference import make_inference, make_ensemble_inference

INFERENCE_FROM_LAST_TRAINED = True
RUN_ENSEMBLE_INFERENCE = True  # Requires ensemble_runs from the training loop

if INFERENCE_FROM_LAST_TRAINED and 'exp_id' in globals():
    model_path = BASE_PATH / 'experiments/models' / f"{exp_id}.pt"
    make_inference(
        loader=test_loader,
        device=device,
        input_shape=input_shape,
        model_path=Path(model_path),
        model_name=MODEL_NAME,
        experiment_id=exp_id,
        base_path=str(BASE_PATH),
    )

if RUN_ENSEMBLE_INFERENCE:
    ensemble_runs = ensemble_runs if 'ensemble_runs' in globals() else []
    if ensemble_runs:
        ensemble_paths = [run['model_path'] for run in ensemble_runs]
        ensemble_names = [run.get('model_name', MODEL_NAME) for run in ensemble_runs]
        make_ensemble_inference(
            loader=test_loader,
            device=device,
            input_shape=input_shape,
            model_paths=ensemble_paths,
            model_names=ensemble_names,
            base_path=str(BASE_PATH),
            inference_type='weighted_majority_vote',
            patch_vote='soft',
            model_weights=None,
            save_prefix='ensemble_5splits',
        )
    else:
        print('No ensemble_runs available. Train the ensemble loop first or disable RUN_ENSEMBLE_INFERENCE.')


[DEBUG] Initializing CNN model with the following parameters:
input_shape: (3, 224, 224)
num_classes: 4
num_blocks: 2
convs_per_block: 1
use_stride: False
stride_value: 2
padding_size: 1
pool_size: 2
initial_channels: 32
channel_multiplier: 2
dropout_rate_classifier_head: 0.2


RuntimeError: DataLoader worker (pid(s) 16467, 16468) exited unexpectedly